In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sqlalchemy import text
from src import config as C
from src.db_utils import run_sql_file

run_sql_file("02_abc_xyz.sql", {
    "first_week": C.FIRST_WEEK, "train_end": C.TRAIN_END,
    "a_cut": C.ABC_THRESHOLDS[0], "b_cut": C.ABC_THRESHOLDS[1],
    "x_cut": C.XYZ_THRESHOLDS[0], "y_cut": C.XYZ_THRESHOLDS[1],
})

engine = C.get_engine()
seg = pd.read_sql(text("SELECT * FROM seg_abc_xyz"), engine)
print(seg.shape)
seg.head()

(500, 8)


,store,item,revenue,cum_share,abc_class,mean_weekly,cv,xyz_class
0,2,45,1238504.22,0.005845,A,701.3365,0.205540,X
1,8,45,1188820.74,0.011456,A,673.2019,0.209557,Y
2,2,12,1179016.22,0.017021,A,606.8894,0.210564,Y
3,2,25,1172199.42,0.022553,A,703.5673,0.208840,Y
4,8,12,1132110.74,0.027897,A,582.7452,0.206828,X


In [2]:
print(seg["abc_class"].value_counts().sort_index(), "\n")
print(seg["xyz_class"].value_counts().sort_index(), "\n")
print(pd.crosstab(seg["abc_class"], seg["xyz_class"]), "\n")
print(seg["cv"].describe())

abc_class
A    277
B    134
C     89
Name: count, dtype: int64 

xyz_class
X    167
Y    167
Z    166
Name: count, dtype: int64 

xyz_class    X    Y   Z
abc_class              
A          123  108  46
B           32   46  56
C           12   13  64 

count    500.000000
mean       0.212039
std        0.006599
min        0.197894
25%        0.207657
50%        0.210662
75%        0.215377
max        0.251311
Name: cv, dtype: float64


In [13]:
x_cut, y_cut = seg["cv"].quantile([1/3, 2/3]).round(4)
print("X/Y cut:", x_cut, " Y/Z cut:", y_cut)

X/Y cut: 0.2086  Y/Z cut: 0.2135
